# Boundary Loss λ3 Sweep — Phase 2 (Person 5's Week 10 task)

Tunes λ3 for the Boundary Dice Loss (proposal §3.4) at full scale (3576/766/766, seed 42, 20 epochs). Fixed at Dinura's λ2-sweep winner (λ2=1.0, MSE, `l2_1_mse`) — every cell here retrains from the ImageNet-pretrained backbone with L_att (λ2=1.0) + L_boundary (λ3 swept), same protocol as Dinura's λ2 sweep. No seeding step needed (unlike that sweep) since there's no completed teammate run to reuse.

Upload **only** `boundary_sweep.zip` (from `make_boundary_colab_zip.py`), not the whole repo.

**Before running:** Runtime → GPU (T4+). Upload the zip to `MyDrive/boundary_sweep.zip`.


## Step 0: Unzip + deps

Checkpoints/results write to `MyDrive/boundary_sweep_outputs` so a dropped runtime does not lose multi-hour runs.


In [ ]:
import sys, zipfile, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
ZIP_ON_DRIVE = Path("/content/drive/MyDrive/boundary_sweep.zip")
BUNDLE = Path("/content/boundary_sweep")
OUTPUTS = Path("/content/drive/MyDrive/boundary_sweep_outputs")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not (BUNDLE / "paths.py").exists():
        if not ZIP_ON_DRIVE.is_file():
            raise FileNotFoundError(f"Upload zip to {ZIP_ON_DRIVE}")
        print("Unzipping", ZIP_ON_DRIVE)
        with zipfile.ZipFile(ZIP_ON_DRIVE, "r") as z:
            z.extractall(BUNDLE.parent)
        if not (BUNDLE / "paths.py").exists():
            cands = [p for p in BUNDLE.parent.iterdir() if p.is_dir() and (p / "paths.py").exists()]
            if not cands:
                raise FileNotFoundError("paths.py not found after unzip")
            if BUNDLE.exists():
                shutil.rmtree(BUNDLE)
            shutil.move(str(cands[0]), str(BUNDLE))
    HERE = BUNDLE
else:
    HERE = Path.cwd()
    if not (HERE / "paths.py").exists():
        cand = Path("Phase2/Dhinanjaya-Person5").resolve()
        if (cand / "run_boundary_sweep.py").exists():
            HERE = cand
    OUTPUTS = HERE

sys.path.insert(0, str(HERE))
print("HERE =", HERE)


In [ ]:
import subprocess, sys
pkgs = ["transformers", "accelerate", "thop", "tqdm", "opencv-python-headless"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("deps ready")


In [ ]:
import paths
from paths import LAYOUT_MODE, PERSON3_DIR, add_teammate_paths, apply_data_dirs, set_output_roots

set_output_roots(OUTPUTS / "checkpoints", OUTPUTS / "results")
add_teammate_paths()
apply_data_dirs()

print("layout:", LAYOUT_MODE)
print("attn pkg:", (PERSON3_DIR / "attention_consistency").is_dir())
print("boundary_refinement pkg:", (HERE / "boundary_refinement").is_dir())
print("images:", paths.DATA_IMG_DIR.is_dir(), paths.DATA_IMG_DIR)
print("masks:", paths.DATA_MASK_DIR.is_dir(), paths.DATA_MASK_DIR)
print("CKPT root:", paths.OUTPUT_ROOT_CKPT)
print("RESULTS root:", paths.OUTPUT_ROOT_RESULTS)
assert (PERSON3_DIR / "attention_consistency").is_dir()
assert (HERE / "boundary_refinement").is_dir()
assert paths.DATA_IMG_DIR.is_dir() and paths.DATA_MASK_DIR.is_dir()


## Step 1: Device check


In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available():
    print("WARNING: no GPU -- each cell is ~2h on T4; CPU will not finish.")


## Step 2: Train the λ3 sweep cells (0.1, 0.2, 0.5)

Fixed λ2=1.0 / MSE (Dinura's winner). Skips any cell whose `segformer_b0_att_best.pt` already exists. ~2 hours per cell on T4.


In [ ]:
import run_boundary_sweep as S
import sys
sys.argv = [
    "run_boundary_sweep.py",
    "--lambda3", "0.1", "0.2", "0.5",
    "--lambda2", "1.0",
    "--att-mode", "mse",
]
S.main()


## Step 3: Eval λ3 cells that have checkpoints


In [ ]:
import eval_boundary_sweep as E
import sys
sys.argv = [
    "eval_boundary_sweep.py",
    "--lambda3", "0.1", "0.2", "0.5",
    "--lambda2", "1.0",
    "--att-mode", "mse",
]
E.main()


## Step 4: Show sweep table + winner


In [ ]:
from aggregate_boundary_sweep import write_boundary_sweep_table, OWN_RESULTS
payload = write_boundary_sweep_table()
print((OWN_RESULTS / "boundary_sweep_comparison.md").read_text())
print("boundary_winning_config.json:")
print((OWN_RESULTS / "boundary_winning_config.json").read_text())


## Step 5: Copy the winning checkpoint + summary table back to the repo

Run this locally (not in Colab) after downloading `MyDrive/boundary_sweep_outputs/` -- copies the winning cell's checkpoint into `Phase2/Dinura-Person3/checkpoints/runs/<tag>/` and the `results/` tree, matching where `train_full_scale.py --lambda3` writes when run directly (not through this sweep). Then run `python aggregate_boundary_sweep.py` locally to refresh `Phase2/Dhinanjaya-Person5/results/boundary_sweep_comparison.md`, and hand the winning row to Lasana's `Phase2/Lasana-Person4/fold_full_scale_results.py` TODO.
